In [27]:
import tweepy
import json
import numpy as np
from collections import defaultdict
import sqlite3
from datetime import date

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
import time

In [2]:
web = 'https://trends24.in/austria/index.html'
path = "/usr/local/bin/chromedriver"

# Trendanalyse
### Durchsuchen der Seite

In [3]:
options = Options()
options.add_argument("--headless")  # Ohne GUI starten
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")

# WebDriver starten
service = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service, options=options)

try:
    # Twitter Login-Seite öffnen
    driver.get(web)
    # Wartezeit-Objekt
    wait = WebDriverWait(driver, 10)

    # 🔹 1. "AGREE"-Button suchen und klicken, falls vorhanden
    try:
        agree_button = wait.until(EC.presence_of_element_located((By.XPATH, "//button[span[text()='AGREE']]")))
        agree_button.click()
        print("Button 'AGREE' wurde erfolgreich geklickt.")
        time.sleep(2)  # Kurz warten, damit Seite weiterlädt
    except:
        print("Kein 'AGREE'-Button gefunden, weiter mit dem nächsten Schritt.")

    # 🔹 2. Versuchen, den "View all 50 trends"-Button zu klicken
    try:
        # Button finden
        view_all_button = wait.until(EC.presence_of_element_located((By.CLASS_NAME, "view-all-btn")))

        # Button per JavaScript klicken
        driver.execute_script("arguments[0].click();", view_all_button)
        print("Button 'View all 50 trends' wurde erfolgreich geklickt.")

        # Warten, bis neue Inhalte erscheinen
        wait.until(EC.presence_of_all_elements_located((By.CLASS_NAME, "list-container")))

    except Exception as e:
        print(f"Fehler beim Klicken auf den Button: {e}")

    # Alle "list-container" Elemente finden
    list_containers = driver.find_elements(By.CLASS_NAME, "list-container")

    # Dictionary zum Speichern der Ergebnisse
    results_dict = {}

    for container in list_containers:
        try:
            # Titel extrahieren (h3)
            title_element = container.find_element(By.TAG_NAME, "h3")
            title_text = title_element.text.strip()

            # Alle "li" Elemente innerhalb des Containers finden
            list_items = container.find_elements(By.CSS_SELECTOR, "ol.trend-card__list > li")

            # Trendnamen aus allen <a> Elementen extrahieren
            trend_names = [item.find_element(By.CSS_SELECTOR, "a.trend-link").text for item in list_items]

            # Ergebnisse ins Dictionary speichern (Titel als Schlüssel, Trends als Wert)
            results_dict[title_text] = trend_names

        except Exception as e:
            print(f"Fehler beim Verarbeiten eines list-container")
            #print(f"Fehler beim Verarbeiten eines list-container: {e}")

    # Ergebnisse ausgeben (als Dictionary)
    #print(results_dict)
    
except Exception as e:
    if len(list_containers) != 24:
        print(f"Fehler beim finden eines list-container: {e}")

# Webdriver schließen
driver.quit()

Button 'AGREE' wurde erfolgreich geklickt.
Button 'View all 50 trends' wurde erfolgreich geklickt.
Fehler beim Verarbeiten eines list-container


### Zeitintervalle

In [4]:
time_list = list(results_dict.keys())

# Die Listen mit den Dictionary-Keys
first_keys = time_list[:1]
second_keys = time_list[1:3]
third_keys = time_list[2:6]
fourth_keys = time_list[6:13]
fifth_keys = time_list[13:]

# Funktion zum Zählen der Häufigkeit aller Werte aus mehreren Dictionaries
def count_occurrences(dict_keys):
    counter = {}
    for key in dict_keys:
        for item in results_dict[key]:  # Durch die Werte der Keys iterieren
            counter[item] = counter.get(item, 0) + 1
    return list(counter.items())  # Tupel-Liste zurückgeben

# Listen mit (Element, Anzahl der Vorkommen)
first_list = count_occurrences(first_keys)
second_list = count_occurrences(second_keys)
third_list = count_occurrences(third_keys)
fourth_list = count_occurrences(fourth_keys)
fifth_list = count_occurrences(fifth_keys)

# Anzahl der verschiedenen Elemente in jeder Liste ausgeben
print(len(first_list))
print(len(second_list))
print(len(third_list))
print(len(fourth_list))
print(len(fifth_list))

50
53
73
74
85


### Gewichtsfunktion

In [ ]:
# Hashtag-Listen aus den verschiedenen Zeiträumen
time_buckets = [first_list, second_list, third_list, fourth_list, fifth_list]

# Exponentielle Gewichtung für langfristige Trends
lambda_decay = 0.2  # Steuert, wie schnell alte Werte abfallen

long_term_trend = defaultdict(float)
short_term_trend = defaultdict(float)

# Langfristige Gewichtung berechnen
for i, bucket in enumerate(time_buckets):
    time_factor = np.exp(-lambda_decay * i)  # Ältere Einträge weniger gewichten
    for hashtag, count in bucket:
        long_term_trend[hashtag] += count * time_factor

# Kurzfristiges Wachstum berechnen
for i in range(1, len(time_buckets)):
    prev_counts = dict(time_buckets[i - 1])
    curr_counts = dict(time_buckets[i])
    
    for hashtag in set(prev_counts.keys()).union(set(curr_counts.keys())):
        prev_count = prev_counts.get(hashtag, 0)
        curr_count = curr_counts.get(hashtag, 0)
        growth = curr_count - prev_count  # Wachstum berechnen
        short_term_trend[hashtag] += growth

# Sortiere die Trends nach Gewichtung
sorted_long_term = sorted(long_term_trend.items(), key=lambda x: x[1], reverse=True)
sorted_short_term = sorted(short_term_trend.items(), key=lambda x: x[1], reverse=True)

In [22]:
def process_hashtag_lists(long_term_list, short_term_list, decimal_places=3, top_n=10):

    # Runde Werte auf gewünschte Anzahl an Nachkommastellen
    rounded_long_term = [(tag, round(weight, decimal_places)) for tag, weight in long_term_list]
    
    # Begrenze die Liste auf die ersten "top_n" Einträge
    trimmed_long_term = rounded_long_term[:top_n]
    trimmed_short_term = short_term_list[:top_n]
    
    return trimmed_long_term, trimmed_short_term

# Funktion ausführen
processed_long_term, processed_short_term = process_hashtag_lists(sorted_long_term, sorted_short_term)

# Ergebnisse ausgeben
print("Processed Long-Term List:", processed_long_term)
print("Processed Short-Term List:", processed_short_term)

Processed Long-Term List: [('#ECR2025', 13.204), ('NGOs', 13.204), ('Julie', 13.204), ('Omas', 13.204), ('Ludwig', 13.204), ('Zivilgesellschaft', 13.204), ('Organisationen', 13.204), ('Transparenz', 12.534), ('Virus', 12.306), ('Wien', 12.204)]
Processed Short-Term List: [('Pandemie', 9.0), ('Wien', 9.0), ('Steuergeld', 9.0), ('Förderung', 9.0), ('Qualifikation', 9.0), ('Transparenz', 8.0), ('Hanke', 8.0), ('Organisationen', 8.0), ('Drittel', 8.0), ('Mieten', 8.0)]


### Aufnahme in Database

In [34]:
# Datenbankverbindung herstellen
DB_PATH = "hashtags.db"

def create_tables():
    """Erstellt die Tabellen für Short-Term- und Long-Term-Hashtags, falls sie noch nicht existieren."""
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()

    # Tabelle für Short-Term-Hashtags (jeden Tag überschreiben)
    cursor.execute("""
    CREATE TABLE IF NOT EXISTS short_term (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        hashtag TEXT UNIQUE,
        weight REAL,
        date TEXT
    );
    """)

    # Tabelle für Long-Term-Hashtags (jeden Tag neue Einträge)
    cursor.execute("""
    CREATE TABLE IF NOT EXISTS long_term (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        hashtag TEXT,
        weight REAL,
        date TEXT
    );
    """)

    conn.commit()
    conn.close()

In [29]:
def update_short_term(short_term_list):
    """Löscht alte Short-Term-Hashtags und fügt die neuen ein."""
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()

    # Lösche alte Short-Term-Hashtags für den heutigen Tag
    cursor.execute("DELETE FROM short_term WHERE date = ?", (date.today().isoformat(),))

    # Neue Einträge hinzufügen
    cursor.executemany(
        "INSERT INTO short_term (hashtag, weight, date) VALUES (?, ?, ?);",
        [(tag, weight, date.today().isoformat()) for tag, weight in short_term_list]
    )

    conn.commit()
    conn.close()

def add_long_term(long_term_list):
    """Fügt die neuen Long-Term-Hashtags hinzu (keine Löschung der alten)."""
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()

    # Neue Einträge hinzufügen
    cursor.executemany(
        "INSERT INTO long_term (hashtag, weight, date) VALUES (?, ?, ?);",
        [(tag, weight, date.today().isoformat()) for tag, weight in long_term_list]
    )

    conn.commit()
    conn.close()

In [45]:
# Tabellen erstellen, falls nicht vorhanden
create_tables()

# Short-Term-Hashtags aktualisieren (alte werden gelöscht, neue eingefügt)
update_short_term(processed_short_term)

# Long-Term-Hashtags hinzufügen (nur neue Einträge)
add_long_term(processed_long_term)

print("Daten erfolgreich aktualisiert!")

Daten erfolgreich aktualisiert!


### Daten aufrufen

In [46]:
def get_short_term():
    """Holt die Short-Term-Hashtags für den heutigen Tag."""
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    cursor.execute("SELECT hashtag, weight FROM short_term WHERE date = ?", (date.today().isoformat(),))
    data = cursor.fetchall()
    conn.close()
    return data

def get_long_term(date_filter=None):
    """
    Holt die Long-Term-Hashtags aus der Datenbank.
    
    :param date_filter: Optionales Datum im Format 'YYYY-MM-DD'. Falls None, werden alle Einträge geholt.
    :return: Liste von (Hashtag, Gewicht, Datum).
    """
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()

    if date_filter:
        cursor.execute("SELECT hashtag, weight, date FROM long_term WHERE date = ?", (date_filter,))
    else:
        cursor.execute("SELECT hashtag, weight, date FROM long_term")

    data = cursor.fetchall()
    conn.close()
    return data

In [47]:
print(get_short_term())

[('Pandemie', 9.0), ('Wien', 9.0), ('Steuergeld', 9.0), ('Förderung', 9.0), ('Qualifikation', 9.0), ('Transparenz', 8.0), ('Hanke', 8.0), ('Organisationen', 8.0), ('Drittel', 8.0), ('Mieten', 8.0)]


In [50]:
# Beispiel: Long-Term-Hashtags für den 25. Februar 2025 abrufen
datum = "2025-02-28"
long_term_specific_day = get_long_term(datum)

# Ausgabe
print(f"Long-Term-Hashtags für den {datum}:", long_term_specific_day)
print(len(get_long_term(datum)))

Long-Term-Hashtags für den 2025-02-28: [('#ECR2025', 13.204, '2025-02-28'), ('NGOs', 13.204, '2025-02-28'), ('Julie', 13.204, '2025-02-28'), ('Omas', 13.204, '2025-02-28'), ('Ludwig', 13.204, '2025-02-28'), ('Zivilgesellschaft', 13.204, '2025-02-28'), ('Organisationen', 13.204, '2025-02-28'), ('Transparenz', 12.534, '2025-02-28'), ('Virus', 12.306, '2025-02-28'), ('Wien', 12.204, '2025-02-28')]
10
